In [0]:
import os

BASE = "s3://mimic-iv-datasets/mimic-iv-dataset-3.1"
HOSP_PATH = f"{BASE}/hosp"
ICU_PATH = f"{BASE}/icu"

CATALOG = "workspace"
BRONZE = "mimic_iv_3_1_bronze"
CATALOG_PATH = f"{CATALOG}.{BRONZE}"

In [0]:
def RetrieveTable(table_name):
    df = spark.table(f"{CATALOG_PATH}.{table_name}")

    print(f"Total rows: {df.count():,}")
    print(f"Total columns: {len(df.columns)}")

    table_info = spark.sql(f"DESCRIBE DETAIL {CATALOG_PATH}.{table_name}").collect()[0]
    table_size_bytes = table_info.sizeInBytes
    table_size_megabytes = table_size_bytes / (1024 * 1024)

    print(f"Table Size: {table_size_megabytes:.2f} MB")

    print(f"\nSchema:")
    df.printSchema()

    print(f"\nSample data:")
    display(df.limit(5))

    return df

In [0]:
patients_df = RetrieveTable("hosp_patients")

In [0]:
chartevents_df  = RetrieveTable("icu_chartevents")

In [0]:
# Get all tables in the bronze schema
tables = spark.sql(f"SHOW TABLES IN {CATALOG_PATH}").collect()

for table_info in tables:
    table_name = table_info.tableName
    full_table_name = f"{CATALOG_PATH}.{table_name}"
    
    print(f"Table: {table_name}")

    df = spark.table(full_table_name)
    
    row_count = df.count()
    print(f"Total Rows: {row_count:,}\n")
    
    print(f"{"Column Name":<40} {"Data Type":<25} {"Null Count":<15} {"Null %":<10}")
    print("-" * 100)
    
    for field in df.schema.fields:
        col_name = field.name
        col_type = str(field.dataType)
        
        # Count nulls for this column
        null_count = df.filter(df[col_name].isNull()).count()
        null_percentage = (null_count / row_count * 100) if row_count > 0 else 0
        
        print(f"{col_name:<40} {col_type:<25} {null_count:<15,} {null_percentage:<10.2f}%")
    
    print()